# Lab Day 10: Finetune Qwen3-VL-4B (LoRA) để định vị hư hại mặt đường
Đọc `README.md` và `GUIDE.md` trước. Notebook này chỉ là **khung**: bạn tự viết mã theo pseudo code trong GUIDE.

## 0. Môi trường (Colab T4)

In [ ]:
# Colab starts in /content; clone the public fork so scripts and configs are available.
from pathlib import Path
import subprocess

REPO_DIR = Path("/content/K4-Track-4-Day10-VisionLanguageModel")
REPO_URL = "https://github.com/Duytd26/K4-Track-4-Day10-VisionLanguageModel.git"
if not (REPO_DIR / ".git").exists():
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(REPO_DIR)], check=True)
%cd /content/K4-Track-4-Day10-VisionLanguageModel


In [ ]:
# Runtime → Change runtime type → T4 GPU
!nvidia-smi -L
!pip -q install -r requirements-colab.txt

In [ ]:
# Run the assignment integrity checks before downloading data.
!python -m pytest -q


In [ ]:
# Mount Drive so adapters and checkpoints survive Colab restarts.
from pathlib import Path
from google.colab import drive

drive.mount("/content/drive")
DRIVE_RUNS = Path("/content/drive/MyDrive/day10-vlm/runs")
DRIVE_RUNS.mkdir(parents=True, exist_ok=True)
print(f"Artifacts will be backed up under: {DRIVE_RUNS}")


In [ ]:
# Kaggle: dùng Colab Secrets (KAGGLE_USERNAME, KAGGLE_KEY). KHÔNG viết khoá vào notebook.
import os
from google.colab import userdata
os.environ['KAGGLE_USERNAME'] = userdata.get('KAGGLE_USERNAME')
os.environ['KAGGLE_KEY'] = userdata.get('KAGGLE_KEY')

## 1. Dữ liệu: tải và chia split

In [ ]:
!python scripts/prepare_data.py --kaggle

In [ ]:
# Verify the fixed session split, compare counts, and inspect six labeled training images.
import json
from collections import Counter
from pathlib import Path
from PIL import Image, ImageDraw
from IPython.display import display
from scripts.labcommon import read_jsonl

DATA_ROOT = Path("data")
SPLIT_DIR = DATA_ROOT / "splits"
with (SPLIT_DIR / "meta.json").open(encoding="utf-8") as f:
    split_meta = json.load(f)
dataset_version = split_meta["dataset_version"]
assert dataset_version.startswith("d10-"), dataset_version

expected = {
    "train": {"images": 1422, "boxes": {"pothole": 1074, "crack": 1750, "manhole": 691}},
    "val":   {"images": 276,  "boxes": {"pothole": 66,   "crack": 470,  "manhole": 103}},
    "test":  {"images": 311,  "boxes": {"pothole": 121,  "crack": 299,  "manhole": 163}},
}
split_rows = {s: read_jsonl(SPLIT_DIR / f"{s}.jsonl") for s in ("train", "val", "test")}
split_sessions = {s: {r["session"] for r in rows} for s, rows in split_rows.items()}
for s, rows in split_rows.items():
    image_count = len(rows)
    box_counts = Counter(o["label"] for r in rows for o in r["objects"])
    assert image_count == expected[s]["images"], (s, image_count)
    assert dict(box_counts) == expected[s]["boxes"], (s, dict(box_counts))
    assert split_meta["counts"][s]["images"] == image_count
    assert split_meta["counts"][s]["boxes"] == dict(box_counts)
    assert all(split_meta["groups"].get(session) == s for session in split_sessions[s])
for left, right in (("train", "val"), ("train", "test"), ("val", "test")):
    overlap = split_sessions[left] & split_sessions[right]
    assert not overlap, f"Session leakage between {left} and {right}: {sorted(overlap)[:5]}"

print(f"dataset_version: {dataset_version}")
print("split  images  pothole  crack  manhole  sessions")
for s, rows in split_rows.items():
    c = split_meta["counts"][s]["boxes"]
    print(f"{s:5}  {len(rows):6}  {c['pothole']:7}  {c['crack']:5}  {c['manhole']:7}  {len(split_sessions[s]):8}")
print("Session sets are disjoint across train/val/test.")

train_samples = split_rows["train"]
colors = {"pothole": "red", "crack": "yellow", "manhole": "cyan"}
chosen, used_sessions = [], set()
for row in sorted(train_samples, key=lambda r: r["image_id"]):
    if row["session"] not in used_sessions and row["objects"]:
        chosen.append(row)
        used_sessions.add(row["session"])
    if len(chosen) == 6:
        break
assert len(chosen) == 6, "Need six labeled training images from distinct sessions."

tile_w, tile_h, caption_h = 320, 180, 28
canvas = Image.new("RGB", (tile_w * 3, (tile_h + caption_h) * 2), "white")
for i, row in enumerate(chosen):
    img = Image.open(DATA_ROOT / row["image_path"]).convert("RGB")
    draw = ImageDraw.Draw(img)
    W, H = img.size
    assert (W, H) == (row["width"], row["height"])
    for obj in row["objects"]:
        xc, yc, bw, bh = obj["yolo"]
        x1 = max(0, min(W - 1, round((xc - bw / 2) * W)))
        y1 = max(0, min(H - 1, round((yc - bh / 2) * H)))
        x2 = max(0, min(W - 1, round((xc + bw / 2) * W)))
        y2 = max(0, min(H - 1, round((yc + bh / 2) * H)))
        color = colors[obj["label"]]
        draw.rectangle((x1, y1, x2, y2), outline=color, width=3)
        draw.text((x1, max(0, y1 - 12)), obj["label"], fill=color, stroke_width=1, stroke_fill="black")
    img = img.resize((tile_w, tile_h))
    x = (i % 3) * tile_w
    y = (i // 3) * (tile_h + caption_h)
    canvas.paste(img, (x, y))
    caption = f"{row['image_id']} | {row['session']}"
    ImageDraw.Draw(canvas).text((x + 4, y + tile_h + 4), caption, fill="black")
display(canvas)


## 2. Nhãn: YOLO → JSON thang 0–1000

YOLO boxes store normalized center and size. Convert each edge to the shared 0-1000 scale, then sort boxes by `(y1, x1)` so target JSON order is stable. Empty annotations become `[]`.

The E0 section below loads the frozen base model in NF4, generates one raw completion for every test image, writes `work/e0.jsonl`, and scores it with the provided script.


In [ ]:
# Convert YOLO labels into deterministic Qwen grounding JSON on the 0-1000 scale.
import json
from scripts.labcommon import CLASSES, read_jsonl


def yolo_to_box(label: str, yolo: list[float]) -> dict:
    if label not in CLASSES:
        raise ValueError(f"Unknown class: {label}")
    xc, yc, w, h = yolo
    coords = [
        round(max(0.0, min(1000.0, (xc - w / 2) * 1000))),
        round(max(0.0, min(1000.0, (yc - h / 2) * 1000))),
        round(max(0.0, min(1000.0, (xc + w / 2) * 1000))),
        round(max(0.0, min(1000.0, (yc + h / 2) * 1000))),
    ]
    x1, y1, x2, y2 = coords
    if x2 <= x1:
        if x1 < 1000:
            x2 = x1 + 1
        else:
            x1 = 999
    if y2 <= y1:
        if y1 < 1000:
            y2 = y1 + 1
        else:
            y1 = 999
    return {"label": label, "bbox_2d": [x1, y1, x2, y2]}


def objects_to_answer(objects: list[dict]) -> str:
    boxes = [yolo_to_box(obj["label"], obj["yolo"]) for obj in objects]
    boxes.sort(key=lambda box: (box["bbox_2d"][1], box["bbox_2d"][0]))
    return json.dumps(boxes, separators=(",", ":"))

PROMPT = (
    "Locate road damage in this image. "
    "Detect only objects of classes: pothole, crack, manhole. "
    "Output only a JSON list of objects with label and bbox_2d coordinates "
    "normalized on a 0-1000 scale: "
    '[{"label":"pothole","bbox_2d":[x1,y1,x2,y2]}]. '
    "If no damage is found, output []."
)

# Inspect one converted answer without needing to load the model.
_example = next(row for row in split_rows["train"] if row["objects"])
print("Example target:", objects_to_answer(_example["objects"]))


## 3. E0: zero-shot

In [ ]:
# Load the untouched Qwen3-VL base model for the E0 zero-shot baseline.
import gc
import json
from pathlib import Path

import torch
from PIL import Image
from transformers import AutoProcessor, BitsAndBytesConfig, Qwen3VLForConditionalGeneration
from scripts.labcommon import read_jsonl

MODEL_ID = "Qwen/Qwen3-VL-4B-Instruct"
assert torch.cuda.is_available(), "Select a Colab GPU runtime before loading the model."
quantization = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
    llm_int8_skip_modules=["visual", "lm_head"],
)
model = Qwen3VLForConditionalGeneration.from_pretrained(
    MODEL_ID,
    quantization_config=quantization,
    dtype=torch.float16,
    device_map={"": 0},
)
model.eval()
processor = AutoProcessor.from_pretrained(MODEL_ID)
processor.tokenizer.padding_side = "left"

test_rows = read_jsonl("data/splits/test.jsonl")
assert len(test_rows) == 311, f"Expected 311 test images, found {len(test_rows)}"
BATCH_SIZE = 4
MAX_NEW_TOKENS = 256
pred_path = Path("work/e0.jsonl")
pred_path.parent.mkdir(parents=True, exist_ok=True)

with pred_path.open("w", encoding="utf-8") as pred_file:
    for start in range(0, len(test_rows), BATCH_SIZE):
        batch = test_rows[start:start + BATCH_SIZE]
        images, conversations = [], []
        for row in batch:
            with Image.open(Path("data") / row["image_path"]) as im:
                image = im.convert("RGB")
            images.append(image)
            conversations.append([{
                "role": "user",
                "content": [
                    {"type": "image", "image": image},
                    {"type": "text", "text": PROMPT},
                ],
            }])
        inputs = processor.apply_chat_template(
            conversations,
            tokenize=True,
            add_generation_prompt=True,
            return_dict=True,
            return_tensors="pt",
            padding=True,
        ).to(model.device)
        inputs.pop("token_type_ids", None)
        with torch.inference_mode():
            generated = model.generate(**inputs, max_new_tokens=MAX_NEW_TOKENS, do_sample=False)
        completion_ids = generated[:, inputs["input_ids"].shape[1]:]
        raw_outputs = processor.batch_decode(
            completion_ids,
            skip_special_tokens=True,
            clean_up_tokenization_spaces=False,
        )
        for row, raw in zip(batch, raw_outputs):
            pred_file.write(json.dumps({"image_id": row["image_id"], "raw": raw}, ensure_ascii=False) + "\n")
        pred_file.flush()
        print(f"E0 predictions: {min(start + len(batch), len(test_rows))}/{len(test_rows)}")
        del images, conversations, inputs, generated, completion_ids, raw_outputs
        torch.cuda.empty_cache()

assert pred_path.exists() and pred_path.stat().st_size > 0
print(f"Saved raw E0 predictions to {pred_path}")
# Release VRAM before the later training cells; keep the lightweight processor.
del model
gc.collect()
torch.cuda.empty_cache()


In [ ]:
!python scripts/score.py --pred work/e0.jsonl --name E0 --out report/evidence/e0.json

In [ ]:
# Read the official score and inspect raw model behavior for the report.
import json
from scripts.labcommon import read_jsonl

with open("report/evidence/e0.json", encoding="utf-8") as f:
    e0_result = json.load(f)
assert e0_result["score_version"] == 1
assert e0_result["split"] == "test"
assert e0_result["n_images"] == 311
assert e0_result["missing_predictions"] == 0
assert e0_result["dataset_version"] == dataset_version
print(f"E0 parse_valid_rate: {e0_result['parse_valid_rate']:.3f}")
print(f"E0 micro F1: {e0_result['micro']['f1']:.3f}")
print("First five raw completions:")
for pred in read_jsonl("work/e0.jsonl")[:5]:
    print(pred["image_id"], repr(pred["raw"]))


## 4. Dựng dữ liệu huấn luyện: chat template và mask loss

In [ ]:
# YOUR CODE HERE: encode_example() và collate() (GUIDE bước 4)

## 5. Setup LoRA / đóng băng cho E1, E2, E3

In [ ]:
# YOUR CODE HERE: apply_setup(model, setup) (GUIDE bước 5)

## 6. Vòng huấn luyện

In [ ]:
# YOUR CODE HERE: vòng train, lưu adapter + run.json (GUIDE bước 6)

## 7. Suy luận và chấm E1–E3

In [ ]:
# YOUR CODE HERE: suy luận test → work/eN.jsonl rồi chạy scripts/score.py với --meta (GUIDE bước 7)

## 8. E4 (ablation tự chọn) và E5 (tuỳ chọn, GRPO)

In [ ]:
# YOUR CODE HERE: GUIDE bước 8 và 9